<a href="https://colab.research.google.com/github/phamlong3539-stack/giongminhanh/blob/main/colab_adam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kiên Đoàn TTS — Adam

> **GPU:** T4 16GB (tự động) · **Thời gian:** ~3 phút lần đầu, ~30s/lần sau

## Hướng dẫn

1. **Runtime** → **Run all** (Ctrl+F9)
2. Nhập văn bản → Click **Tạo giọng nói**
3. Nghe + tải file WAV về

> Powered by OmniVoice (MIT License) — github.com/k2-fsa/OmniVoice


In [1]:
print('Dang cai dat (~1 phut)...')
!pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4"
print('Cai dat hoan tat!')

# Tai giong mau 10s tu voice-notebooks repo
!wget -q https://raw.githubusercontent.com/doanquangkien/voice-notebooks/main/samples/adam.mp3 -O voice_sample.mp3
print('Da tai voice sample!')

Dang cai dat (~1 phut)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 86.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ml-dtypes 0.6.0 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 2.0.2 which is incompatible.
jax 0.11.1 requires numpy>=2.1, but you have numpy 2.0.2 which is incompatible.
tifffile 2026.8.23 requires numpy>=2.1, but you have numpy 2.0.2 which is incompatible.
dm-tree 0.1.10 require

In [2]:
print('Đang khởi động Omnivoice... (lần đầu ~5 phút, lần sau ~30 giây)')

import logging, os, re, time
import numpy as np
import torch
import gradio as gr

# Patch: torch._utils removed in torch 2.13+ — alias to torch._C._utils
import torch as _torch
if not hasattr(_torch, '_utils'):
    _torch._utils = _torch._C._utils

# Shim: AutoFeatureExtractor removed in transformers 5.x
# Reads sampling_rate from model config to avoid broken AutoProcessor import
import transformers as _tf
class _SafeAutoFeatureExtractor:
    @staticmethod
    def from_pretrained(model_name, **kwargs):
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True, **kwargs)
            sr = getattr(cfg, 'sampling_rate', 24000)
        except Exception:
            sr = 24000
        class _Result:
            sampling_rate = sr
        return _Result()
_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Wait for GPU
for i in range(30):
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        break
    time.sleep(1)
else:
    print('GPU not available — make sure Runtime > Change runtime type > T4 GPU')

# Load model
DEVICE = get_best_device()
logger.info(f'Loading OmniVoice on {DEVICE}...')
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
)
SAMPLING_RATE = model.sampling_rate
logger.info(f'Model ready — SR: {SAMPLING_RATE}Hz')

# Voice clone prompt
logger.info('Creating VoiceClonePrompt...')
VOICE_PROMPT = model.create_voice_clone_prompt(ref_audio='voice_sample.mp3')
logger.info('Voice prompt ready — Adam')

# Generate function
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=32, guidance_scale=1.8,
    denoise=True, preprocess_prompt=True, postprocess_output=True,
    position_temperature=5.0, class_temperature=0.2,
    pad_duration=0.1, fade_duration=0.1,
)

def generate_voice(text: str):
    text = text.strip()
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    if not paragraphs:
        return None
    if len(paragraphs) == 1:
        audio = model.generate(
            text=paragraphs[0], voice_clone_prompt=VOICE_PROMPT,
            language='vi', speed=0.95, generation_config=GEN_CFG
        )[0]
    else:
        audios = []
        for i, p in enumerate(paragraphs):
            a = model.generate(
                text=p, voice_clone_prompt=VOICE_PROMPT,
                language='vi', speed=0.95, generation_config=GEN_CFG
            )[0]
            audios.append(a)
            if i < len(paragraphs) - 1:
                audios.append(np.zeros(int(SAMPLING_RATE * 0.3)))
        audio = np.concatenate(audios)
    waveform = (audio * 32767).astype(np.int16)
    return (SAMPLING_RATE, waveform)

# Launch UI
print('Khởi động giao diện Kiên Đoàn TTS — Adam...')
gr.close_all()
CSS = ".gradio-container{{max-width:720px!important;margin:0 auto!important;padding:16px!important}}footer{{display:none!important}}"
THEME = gr.themes.Soft(primary_hue='indigo')
with gr.Blocks(title='Kiên Đoàn TTS — Adam', theme=THEME, css=CSS) as demo:
    gr.Markdown('# Kiên Đoàn TTS\\nAdam · Giọng nam cuốn hút, phong cách đối thoại tự tin, phù hợp video ngắn viral, TikTok, Reels.')
    t = gr.Textbox(label='Nhập văn bản', lines=4, placeholder='Nhập văn bản bạn muốn chuyển thành giọng nói...')
    btn = gr.Button('Tạo giọng nói', variant='primary')
    out = gr.Audio(label='Kết quả')
    btn.click(generate_voice, inputs=[t], outputs=[out], concurrency_limit=1)

demo.launch(server_name='0.0.0.0', share=True, theme=THEME, css=CSS)


Đang khởi động Omnivoice... (lần đầu ~5 phút, lần sau ~30 giây)
GPU: Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

Khởi động giao diện Kiên Đoàn TTS — Adam...


/tmp/ipykernel_853/3725915398.py:96: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title='Kiên Đoàn TTS — Adam', theme=THEME, css=CSS) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5b05a67f603721526b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
